# Silver Layer — Order Items Transformation

Transforms `bronze.orders_items` into `silver.order_items`. Filters nulls, deduplicates by composite key `(order_id, order_item_id)`, and validates with four DQ checks including value-range validation for price and freight.

No quarantine flow — pre-check showed zero source violations, so a gate is sufficient.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%run ../Utilities/silver_dq_checks

# Silver Layer — Data Quality Checks

This notebook defines reusable data quality check functions used across the Silver layer transformation notebooks.

Each function validates a specific property of a transformed DataFrame before it is written to a Silver Delta table. On success the function prints a PASSED message; on failure it raises a `ValueError` with details, stopping the pipeline.

## Checks Defined Here

* `check_not_null` — verifies that specified columns have no null or blank values
* `check_unique` — verifies that the given key (single or composite) has no duplicates
* `check_row_count` — verifies that the Silver row count falls within an expected percentage range of the Bronze source

## How To Use

Import these functions into a Silver transformation notebook by running:
​```
%run ../Utilities/silver_dq_checks
​```
Then call the functions on the transformed DataFrame before writing to Silver.

### check_not_null
Validates that the specified columns contain no null or blank/whitespace values.

In [0]:
catalog_name = 'retail_lakehouse'
bronze_schema = 'bronze'
silver_schema = 'silver'
quarantine_schema = 'quarantine'

### Transformations

In [0]:
df_order_items = spark.read.table(f'{catalog_name}.{bronze_schema}.orders_items')
transformed_items_df = df_order_items.filter((col('order_id').isNotNull()) 
                                 & (col('order_item_id').isNotNull()) 
                                 & (col('product_id').isNotNull()) 
                                 & (col('seller_id').isNotNull()) 
                                 & (col('shipping_limit_date').isNotNull()) 
                                 & (col('price').isNotNull())
                                  & (col('freight_value').isNotNull()))\
                                  .dropDuplicates(['order_id','order_item_id'])\
                                      .withColumn('silver_processed_at',current_timestamp())

### check_unique
Validates that the specified key columns produce unique rows. Supports single-column or composite keys.

### check_row_count
Validates that the Silver row count is within an acceptable percentage range of the Bronze source row count.


### Data quality checks

In [0]:
check_not_null(transformed_items_df, ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value'])
check_unique(transformed_items_df,['order_id','order_item_id'])
check_row_count(transformed_items_df,f'{catalog_name}.{bronze_schema}.orders_items',85,100)
check_value_range(transformed_items_df,[{'column':'price','min_value':0,'inclusive': False},{'column':'freight_value','min_value':0,'inclusive':True}])


check_not_null: PASSED
check_unique: PASSED
check_row_count: PASSED
check_value_range: PASSED


### check_value_range
Check if the columns follows rules defined in check_value_range.

### Writing clean dataframe to silver schema

## Writing tranformed dataframe to silver schema

In [0]:
transformed_items_df.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.{silver_schema}.order_items')
